# DATA INGESTION
This notebook is for the data ingestion of crm_sales. 

Crm Sales has a total of 4 tables: 
- accounts
- product
- sales pipeline
- sales team

The following ingestion can be used for... but should not be used for.....


In [0]:
%sql
USE CATALOG `crm_sales`;

CREATE SCHEMA IF NOT EXISTS crm_raw;
CREATE VOLUME IF NOT EXISTS crm_raw.source_files;

In [0]:
catalog = "crm_sales"
schema = "crm_raw"

source_path = f"/Volumes/{catalog}/{schema}/source_files"

table_names = [
    "accounts",
    "data_dictionary",
    "sales_teams",
    "sales_pipeline",
    "products"
]

for table_name in table_names:

    # Read this CSV, keeping the columns as text for now
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .csv(f"{source_path}/{table_name}.csv"))

    target_table = f"`{catalog}`.`{schema}`.`{table_name}`"

    source_rows = df.count()

    # Store the data as a persistent Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table))

    # Check that the saved row count matches
    saved_rows = spark.table(target_table).count()
    assert source_rows == saved_rows, f"Row mismatch: {table_name}"
    print(f"{table_name}: {saved_rows:,} rows saved")

In [0]:
%sql
--displaying a table
SELECT 
*
FROM
crm_sales.crm_raw.accounts

In [0]:
%sql
SELECT 
*
FROM 
crm_sales.crm_raw.accounts